# Activity C: Grounded Live-Corpus
### ISA Tutorial — CHIIR 2026

---

## Objective

In this notebook you will see why some questions simply **cannot** be answered from a fixed document set or a structured API — and how **live web retrieval** (Level 4 on the Complexity Ladder) fills that gap.

We focus on **Grounded Live-Corpus**: the LLM searches the open web in real time, reads the top results, synthesizes an answer, and cites its sources.

| Level | Name | What the system needs |
|-------|------|-----------------------|
| 2 | Structured Lookup | Single value from a deterministic API |
| 3 | Grounded Closed-Corpus | Fixed document set (RAG); answer cited from that corpus |
| **4** | **Grounded Live-Corpus** | **Open web / live search for freshness; cite sources; one-shot retrieval + synthesis** |
| 5 | Agentic Navigation | Multi-step tool use; agent decides what to retrieve next |

---

## What you will see

**Part 1 — The LLM alone fails on live questions**
1. Ask the LLM about this week's AI policy news with no context → stale or hedged answer.

**Part 2 — Fixing it with DuckDuckGo**
1. Use [DuckDuckGo](https://docs.searxng.org/) — a free, open-source metasearch engine — to retrieve live web results (no API key, no account).
2. Inject the top results into the prompt → LLM gives a current, cited answer.
3. Repeat for three diverse live questions to build intuition.

**Part 3 — Why not a closed corpus or a structured API?**
1. Show the same live question failing on a static corpus.
2. Understand when Level 4 is the right choice vs. Levels 2 and 3.

**Part 4 — Source quality and the new challenges of the open web**
1. Inspect which search engines contributed each result.
2. Handle a query where sources disagree.

---

> **Runtime note:** The default model is `Qwen/Qwen2-0.5B-Instruct` (0.5 B params, ~1 GB download).  
> It runs comfortably on a laptop CPU. On Colab with a T4 GPU, swap to  
> `microsoft/Phi-3.5-mini-instruct` for richer answers — see the comment in the model cell.  
> The only other dependency is `requests` — no extra ML libraries needed for this activity.

In [ ]:
# ── Install dependencies ────────────────────────────────────────────────────
# transformers       : HuggingFace model loading & text generation
# torch              : tensor operations (CPU or GPU)
# duckduckgo-search  : free, keyless web search — no signup required

!pip install -q transformers torch duckduckgo-search

## Setting up the LLM

Same setup as Activity B. We load **Qwen2-0.5B-Instruct** once and reuse a single `generate(prompt)` helper throughout.

> **Want a stronger model?** On a Colab T4 GPU, swap `MODEL_ID` to  
> `"microsoft/Phi-3.5-mini-instruct"` (3.8 B params). The core lesson is identical.

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# ── Model selection ──────────────────────────────────────────────────────────
MODEL_ID = "Qwen/Qwen2-0.5B-Instruct"          # 0.5 B params, ~1 GB, works on CPU
# GPU upgrade (Colab T4 or better):
# MODEL_ID = "microsoft/Phi-3.5-mini-instruct"  # 3.8 B params, needs ~4 GB VRAM

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
print(f"Loading {MODEL_ID} ...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16 if device == "cuda" else torch.float32,
)
model = model.to(device)
model.eval()
print("Model ready!")


# ── Helper: generate ────────────────────────────────────────────────────────
def generate(prompt: str, max_new_tokens: int = 300) -> str:
    """
    Send a plain-text prompt to the LLM and return the response string.
    Uses the model's chat template for instruction-following format.
    Greedy decoding (do_sample=False) gives deterministic output.
    """
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

/Users/preetams/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device : cpu
Loading Qwen/Qwen2-0.5B-Instruct ...
Model ready!


---
---

# Part 1 — The LLM Alone Fails on Live Questions

## What is Grounded Live-Corpus?

A **Grounded Live-Corpus** query requires information that is:
- **Temporally fresh** — the answer changes day to day (news, policy updates, software releases, events)
- **Not a single structured value** — it requires reading and synthesizing across multiple web documents
- **Not in a fixed corpus** — no pre-curated document set covers all possible live topics

**Key properties:**
- The "corpus" is the live open web — it changes every second.
- Retrieval is **one-shot**: search → read top-k results → synthesize.
- The LLM must **cite sources** so the answer can be verified.
- Evaluation requires checking freshness, source credibility, and answer faithfulness.

**How it differs from neighbors on the ladder:**

| | Level 2 (Structured Lookup) | Level 3 (Closed-Corpus) | **Level 4 (Live-Corpus)** |
|---|---|---|---|
| Source | Deterministic API | Fixed document set | Open web |
| Freshness | Handled by provider | Fixed at index time | Always current |
| Output | Single value | Extracted passage | Multi-source synthesis |
| Evaluation | Compare to API value | Retrieval hit + faithfulness | Freshness + credibility + faithfulness |

In [ ]:
# ── 1.1  LLM-only attempt: no search context ────────────────────────────────
# The 2026 Oscars ceremony was held in March 2026 — right now.
# The model's training cutoff predates this, so it cannot know the winner.

question = "Who won Best Picture at the 2026 Oscars?"

print("QUESTION:", question)
print("\n" + "─" * 60)
print("LLM answer (no search context):")
print("─" * 60)
llm_only_answer = generate(question)
print(llm_only_answer)


### What just happened?

The LLM has a **training cutoff** — a date beyond which it has seen no data. For anything after that date, it can only:

1. **Hedge** — "I don't have access to real-time information" (honest but useless)
2. **Give stale information** — recall something from training that is no longer current
3. **Hallucinate** — invent plausible-sounding but false specifics

Even if the model were retrained last week, it still cannot see today's news. This is a structural limitation of static model weights, not a quality problem.

> **The fix is not a better model — it is live retrieval.**

---
---

# Part 2 — Fixing it with DuckDuckGo Search

## Why DuckDuckGo?

[DuckDuckGo](https://duckduckgo.com) is a privacy-focused search engine that indexes the live web. The [`duckduckgo-search`](https://pypi.org/project/duckduckgo-search/) Python library provides a clean wrapper with **no API key and no account required**.

```python
from duckduckgo_search import DDGS

with DDGS() as ddgs:
    results = list(ddgs.text("query here", max_results=5))
```

Each result contains: `title`, `href` (URL), and `body` (snippet).

Unlike a fixed corpus, DuckDuckGo always returns **live, current results** — making it ideal for questions about events happening right now.

In [ ]:
from duckduckgo_search import DDGS
from datetime import datetime


def make_search_query(question: str) -> str:
    """
    Distil a natural-language question into a concise search phrase (4-6 words).

    Sending a full sentence to a search engine returns noisy results and may
    trigger rate-limiting. A short phrase is how people actually use search.

    Example:
      question → "Who won Best Picture at the 2026 Oscars?"
      phrase   → "2026 Oscars Best Picture winner"
    """
    prompt = (
        "Convert the following question into a short web search query (4-6 words). "
        "Output ONLY the search phrase — no explanation, no punctuation.\n\n"
        f"Question: {question}\n"
        "Search phrase:"
    )
    raw    = generate(prompt, max_new_tokens=15).strip()
    phrase = raw.splitlines()[0].strip().strip('"').strip("'")
    print(f"  Question : {question}")
    print(f"  → Query  : {phrase}")
    return phrase


def web_search(query: str, top_k: int = 5) -> list:
    """
    Search the live web via DuckDuckGo. No API key. No account.

    Returns a list of dicts with keys: title, url, snippet.
    """
    try:
        with DDGS() as ddgs:
            raw = list(ddgs.text(query, max_results=top_k))
        results = [
            {
                "title":   r.get("title", "").strip(),
                "url":     r.get("href", ""),
                "snippet": r.get("body", "").strip(),
                "engine":  "DuckDuckGo",
            }
            for r in raw
        ]
        print(f"  [DDG] {len(results)} results ✓")
        return results
    except Exception as e:
        print(f"  [DDG] Search failed: {type(e).__name__}: {e}")
        return []


def print_results(results: list) -> None:
    if not results:
        print("No results.")
        return
    for i, r in enumerate(results, 1):
        print(f"\n[{i}] {r['title']}")
        print(f"    {r['url']}")
        print(f"    {r['snippet'][:200]}...")

In [ ]:
# ── 2.1  Run a live search ───────────────────────────────────────────────────
# Step 1: LLM distils the question into a short search phrase.
# Step 2: DuckDuckGo fetches live results for that phrase.

print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

search_phrase = make_search_query(question)
print()

results = web_search(search_phrase, top_k=5)
print_results(results)

In [ ]:
# ── Helper: format results into a grounded prompt ───────────────────────────
def format_as_context(results: list) -> str:
    """
    Format search results as a numbered context block for the LLM.
    Each entry includes title, URL, and snippet so the LLM can cite sources.
    """
    if not results:
        return "[No search results available.]"
    lines = []
    for i, r in enumerate(results, 1):
        lines.append(
            f"[Result {i}]\n"
            f"Title: {r['title']}\n"
            f"URL: {r['url']}\n"
            f"Snippet: {r['snippet']}"
        )
    return "\n\n".join(lines)


def grounded_live_prompt(question: str, results: list) -> str:
    """
    Build a grounded QA prompt from live search results.

    The LLM is instructed to:
      1. Answer ONLY from the provided web results.
      2. Cite the result number and URL for every claim.
      3. Say "Not found in provided results" if snippets don't support an answer.
    This enforces grounding — hallucinations should fail the 'cite your source' test.
    """
    context = format_as_context(results)
    return (
        "You are a research assistant. Answer the question using ONLY the web search results below.\n"
        "For every factual claim, cite the result number and URL (e.g. [Result 2] https://...).\n"
        "If the results do not contain enough information, say \"Not found in provided results.\"\n"
        "\n--- WEB SEARCH RESULTS ---\n"
        f"{context}\n"
        "--------------------------\n"
        f"\nQuestion: {question}\nAnswer:"
    )


In [ ]:
# ── 2.2  LLM with live search context ───────────────────────────────────────
# We inject the live search results into the prompt and ask the same question.

prompt = grounded_live_prompt(question, results)

print("─" * 60)
print("LLM answer (grounded in live search results):")
print("─" * 60)
live_answer = generate(prompt)
print(live_answer)

### Part 2 — Observations

| | LLM-only (no context) | LLM + live search |
|---|---|---|
| Answer currency | Training-cutoff bound | Current as of search time |
| Citations | None | Result number + URL |
| Verifiability | None — must trust model | Click the URL to verify |
| What if wrong? | Indistinguishable from correct | Source can be inspected |

**Key point:** The LLM is doing the same thing in both cases — predicting tokens. The difference is *what information it was given*. Live retrieval turns a hallucination problem into a reading-comprehension problem.

> **Why DuckDuckGo over a single search API?**  
> DuckDuckGo aggregates 240+ search engines simultaneously. No single API gives you this breadth.  
> It is free, open-source, self-hostable, and requires no API key.

In [ ]:
# ── 2.3  Three more live questions ───────────────────────────────────────────
# These showcase the breadth of what live-corpus handles — topics that change
# constantly and span current events, recent awards, and evolving policy.

live_questions = [
    "What are the most recent AI research papers published in March 2026?",
    "Who won Best Picture at the 2026 Oscars?",
    "What is the latest news about the EU AI Act implementation and compliance deadlines?",
]

for i, q in enumerate(live_questions, 1):
    print("=" * 70)
    print(f"Live Question {i}: {q}")
    print("=" * 70)

    res = web_search(q, top_k=4)

    print("\nTop result snippets:")
    for j, r in enumerate(res[:2], 1):   # show only top-2 snippets to keep output readable
        print(f"  [{j}] {r['title']} | {r['url']}")
        print(f"       {r['snippet'][:150]}...")

    answer = generate(grounded_live_prompt(q, res))
    print(f"\nLLM answer:\n{answer}\n")

---
---

# Part 3 — Why Not a Closed Corpus or a Structured API?

## When Level 4 is the right choice

The table below shows why lower-level approaches **structurally fail** on live-information questions — not because of model quality, but because of what information they have access to:

| Approach | Why it fails on live questions |
|----------|--------------------------------|
| LLM alone (Level 1) | Training cutoff; static weights cannot know today's news |
| Structured API (Level 2) | Returns a single clean value (temperature, rate); cannot synthesize a news narrative |
| Closed corpus (Level 3) | The document set was fixed at index time; today's events are not in it |
| **Live-corpus search (Level 4)** | Retrieves whatever is on the web right now; synthesizes across heterogeneous sources |

**Level 4 is the right choice when a question is:**
- **Time-sensitive** (today, this week, most recent)
- **Not a single value** (requires reading and synthesizing multiple documents)
- **Open-domain** (no pre-curated corpus covers the topic)

In [ ]:
# ── 3.1  Demo: the same live question fails on a closed static corpus ─────────
# We simulate a closed corpus — a small, fixed set of documents indexed in the past.
# This mimics what a RAG system over a curated knowledge base would have access to.

STATIC_CORPUS = [
    "The EU AI Act was formally adopted by the European Parliament in March 2024. "
    "It establishes a risk-based framework categorizing AI systems as minimal, limited, "
    "high, or unacceptable risk. High-risk systems require conformity assessments.",

    "Large language models such as GPT-4, Claude, and Gemini are general-purpose AI systems. "
    "They are trained on large text corpora and fine-tuned for instruction following. "
    "They have a knowledge cutoff date beyond which they have no information.",

    "Retrieval-Augmented Generation (RAG) combines a retrieval step with a language model. "
    "A retriever fetches relevant passages from a corpus; the LLM then reads those passages "
    "to generate a grounded answer. Quality depends on retrieval precision.",

    "The ACM SIGIR Conference on Human Information Interaction and Retrieval (CHIIR) "
    "is an annual academic conference focused on user-centered information access, "
    "evaluation of search systems, and information-seeking behavior.",

    "The Oscars, formally the Academy Awards, are presented annually by the Academy of "
    "Motion Picture Arts and Sciences. The Best Picture award is the most prestigious category.",
]


def keyword_search_corpus(query: str, corpus: list, top_k: int = 2) -> list:
    """
    Naive keyword search over a static corpus (simulates a closed RAG system).
    Returns the passages with the most query-word overlap.
    """
    query_words = set(query.lower().split())
    scored = [
        (passage, sum(1 for w in query_words if w in passage.lower()))
        for passage in corpus
    ]
    scored.sort(key=lambda x: x[1], reverse=True)
    return [p for p, score in scored[:top_k] if score > 0]


# Try the same live question against our static corpus
live_q = "What are the most significant AI policy or regulation developments in the past few days?"

print(f"Query: {live_q}")
print(f"Corpus size: {len(STATIC_CORPUS)} documents (indexed in the past)\n")

retrieved = keyword_search_corpus(live_q, STATIC_CORPUS)

if retrieved:
    print("Best matching passage(s) from static corpus:")
    for i, p in enumerate(retrieved, 1):
        print(f"  [{i}] {p[:150]}...")
    print()
    context_block = "\n\n".join(f"[Passage {i+1}]: {p}" for i, p in enumerate(retrieved))
    static_prompt = (
        f"Answer using ONLY the passages below. Cite passage numbers.\n\n"
        f"{context_block}\n\nQuestion: {live_q}\nAnswer:"
    )
    print("LLM answer (static corpus):")
    print(generate(static_prompt))
else:
    print("No relevant passages found in static corpus.")
    print("The corpus has no information about events from the past few days — it is frozen in time.")

### What the closed-corpus result shows

Even if the corpus retriever finds a partially relevant passage (e.g., something about AI regulation), it cannot answer a question about **the past few days** because that information did not exist when the corpus was indexed.

The failure is silent: the LLM may produce a plausible-sounding answer from the stale passage — but it will be **wrong about timing and specifics**.

**The correct diagnosis:** this question has a *freshness* requirement that closed-corpus cannot satisfy. The moment you see "today", "this week", "latest", "most recent" — those are signals that Level 4 (or higher) is needed.

---
---

# Part 4 — Source Quality and the New Challenges of the Open Web

## The open web is noisy

A closed corpus is curated by an expert — every document was deliberately chosen. The open web is not curated: news sites, blogs, Wikipedia edits, press releases, and social media all appear in search results.

This introduces challenges that do not exist at Levels 2 or 3:

| Challenge | Why it matters |
|-----------|----------------|
| **Source heterogeneity** | A blog and a government press release have very different authority |
| **Contradictory snippets** | Different sources may disagree; the LLM must handle or flag conflicts |
| **Staleness within results** | A cached result from three days ago may appear alongside today's news |
| **Missing publication dates** | Not all pages include a date; recency is hard to verify |

Evaluation at Level 4 must check three things:  
1. **Freshness** — are the sources current?  
2. **Source credibility** — who wrote this, and is it authoritative?  
3. **Answer faithfulness** — did the LLM accurately represent what the sources say?

In [ ]:
# ── 4.1  Inspect source diversity ───────────────────────────────────────────
# DuckDuckGo pulls from many domains in a single query. Looking at the domains
# tells us how varied the sourcing is — official docs, news sites, blogs, etc.

from urllib.parse import urlparse

diversity_query = "What is the latest stable release of Python?"
print(f"Query: {diversity_query}\n")

diversity_results = web_search(diversity_query, top_k=8)

print(f"{'Domain':<35} {'Title'}")
print("─" * 80)
for r in diversity_results:
    domain = urlparse(r['url']).netloc.replace('www.', '')
    print(f"{domain:<35} {r['title'][:44]}")

print()
domains = [urlparse(r['url']).netloc.replace('www.', '') for r in diversity_results]
unique  = list(dict.fromkeys(domains))
print(f"Unique domains ({len(unique)}): {unique}")
print()
print("Grounded answer:")
print(generate(grounded_live_prompt(diversity_query, diversity_results[:5])))


In [ ]:
# ── 4.2  A deliberately tricky query: sources may disagree ──────────────────
# Python version releases are a good example: there are often multiple branches
# (e.g. 3.12.x stable vs. 3.13 beta vs. 3.14 alpha) and different pages
# describe different things as "latest".
#
# This mimics real-world ambiguity: news sources update at different speeds,
# old cached pages appear alongside fresh ones, and the LLM must navigate conflict.

tricky_query = "Is Python 3.14 officially released and stable yet?"
print(f"Query: {tricky_query}\n")

tricky_results = web_search(tricky_query, top_k=5)

print("Raw snippets (inspect for disagreements):")
for i, r in enumerate(tricky_results, 1):
    print(f"  [{i}] {r['title']}")
    print(f"       {r['snippet'][:200]}")
    print(f"       Published: {r['published']}  |  Source: {r['url'][:60]}")
    print()

# Prompt the LLM to acknowledge uncertainty if sources conflict
conflict_aware_prompt = grounded_live_prompt(tricky_query, tricky_results) + (
    "\n\nIMPORTANT: If sources disagree, clearly state the disagreement and "
    "explain which source appears most authoritative and why."
)

print("─" * 60)
print("LLM answer (conflict-aware prompt):")
print("─" * 60)
print(generate(conflict_aware_prompt))

### Part 4 — Key takeaways

**1. The engine field matters.**  
Results from `wikipedia` have a different authority profile than results from `google` or a random blog. Auditing which engines contributed a result is part of evaluating a live-corpus answer.

**2. Contradictory sources are normal, not exceptional.**  
The open web does not have a single authoritative version of events. A good live-corpus system must detect conflicts and surface them to the user rather than silently picking one.

**3. Prompt design matters at Level 4.**  
Instructing the LLM to cite, state dates, and flag disagreements is not optional — it is what separates a grounded answer from a confident-sounding hallucination dressed in citation clothes.

**4. Latency is a real cost.**  
Every live-corpus query adds a network round-trip (search) before generation. This is typically 1–5 seconds. At Level 5 (Agentic Navigation), multiple search rounds compound this.

**5. Levers to tune at Level 4:**
- Number of results (`top_k`)
- Which search engines to enable in DuckDuckGo
- Snippet length passed to the LLM
- Prompt framing for conflict resolution and citation style

---
---

# Part 5 — Complexity Ladder Recap

| Level | Name | Information source | Freshness | Synthesis | Evaluation |
|-------|------|--------------------|-----------|-----------|------------|
| 2 | **Structured Lookup** | API / deterministic service | Handled by provider | Minimal — read and report | Compare answer to API ground truth |
| 3 | **Grounded Closed-Corpus** | Fixed document set (RAG) | Fixed at index time | Extraction / short synthesis | Retrieval hit rate + answer faithfulness |
| **4** | **Grounded Live-Corpus** | **Open web (live search)** | **Always current** | **One-shot retrieval + multi-source synthesis** | **Freshness + source credibility + faithfulness** |
| 5 | Agentic Navigation | Multi-step tool use | Always current | Iterative — agent decides what to retrieve next | Trace-aware: each step is evaluated |

---

## What distinguishes Level 4 from its neighbors?

**Level 3 → 4 transition: from fixed to live**  
At Level 3, you hand the agent a corpus and it retrieves from that fixed set. At Level 4, the "corpus" is the entire web at this moment — the agent must choose what to search for and must evaluate source quality on the fly.

**Level 4 → 5 transition: from one-shot to iterative**  
At Level 4, there is one search round: query → top-k results → answer. At Level 5, the agent decides mid-task whether its results were sufficient, formulates follow-up queries, visits pages, interacts with forms, and builds its answer across multiple steps.

---

## What comes next

**Level 5 — Agentic Navigation**  
Multi-step tool use: the agent plans, retrieves, reads intermediate results, decides what to look up next, and may interact with websites or fill forms. Examples: checking campsite availability on a booking site, verifying visa eligibility by navigating multiple government pages.

**Level 6 — Corpus Sensemaking**  
Dataset-wide synthesis: instead of answering a single question, the agent must characterize patterns, themes, or trends across hundreds or thousands of documents. Example: summarizing the main policy positions across all EU AI regulation documents published in a given year.

---

*ISA Tutorial — Information Seeking in the Age of Agentic AI | CHIIR 2026*